# Predictive Churn Model & Risk Scoring

Build machine learning models to predict customer churn probability, score risk, segment customers by risk tier, and recommend targeted interventions to prevent churn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, auc, precision_recall_curve, f1_score, 
                             accuracy_score, precision_score, recall_score)
import joblib

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load and Prepare Data

In [ ]:
# Load data
df = pd.read_csv('../data/telco_churn.csv')

# Data cleaning
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Fill missing TotalCharges (11 values) with 0 (new customers)
df['TotalCharges'].fillna(0, inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['Churn'].mean()*100:.2f}%")
print(f"Missing values: {df.isnull().sum().sum()}")

## 2. Feature Engineering

In [ ]:
# Create engineered features
df_model = df.copy()

# 1. Service adoption count
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                'StreamingTV', 'StreamingMovies', 'PhoneService']
df_model['num_services'] = (df_model[service_cols] == 'Yes').sum(axis=1)

# 2. Has support services (key retention feature from EDA)
df_model['has_support'] = ((df_model['TechSupport'] == 'Yes') | 
                           (df_model['OnlineSecurity'] == 'Yes')).astype(int)

# 3. Tenure buckets (early churn problem)
def tenure_risk(t):
    if t < 6:
        return 3  # High risk
    elif t < 12:
        return 2  # Medium risk
    elif t < 24:
        return 1  # Low risk
    else:
        return 0  # Very low risk
        
df_model['tenure_risk'] = df_model['tenure'].apply(tenure_risk)

# 4. Monthly charges quartile (spending level)
df_model['spending_quartile'] = pd.qcut(df_model['MonthlyCharges'], q=4, labels=[1, 2, 3, 4], duplicates='drop')

# 5. Engagement score (combined metric)
df_model['engagement_score'] = (
    (df_model['num_services'] / 7) * 0.4 +  # Service adoption: 40% weight
    (df_model['tenure'] / df_model['tenure'].max()) * 0.3 +  # Tenure: 30% weight
    ((100 - df_model['MonthlyCharges']) / 100) * 0.3  # Lower charges = higher engagement: 30% weight
) * 100

print("Feature Engineering Complete")
print(f"\nNew features created:")
print(f"- num_services: {df_model['num_services'].describe()}")
print(f"- has_support: {df_model['has_support'].value_counts()}")
print(f"- tenure_risk distribution: {df_model['tenure_risk'].value_counts().sort_index()}")
print(f"- engagement_score: {df_model['engagement_score'].describe()}")

## 3. Prepare Features for Modeling

In [ ]:
# Encode categorical variables
df_encoded = df_model.copy()
label_encoders = {}

categorical_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                        'PaperlessBilling', 'PaymentMethod']

for col in categorical_features:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

# Select features for modeling
feature_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
                'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                'PaperlessBilling', 'PaymentMethod', 'num_services', 'has_support',
                'tenure_risk', 'spending_quartile', 'engagement_score']

X = df_encoded[feature_cols]
y = df_encoded['Churn']

# Encode spending quartile
X['spending_quartile'] = pd.Categorical(X['spending_quartile']).codes

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Churn rate in training: {y_train.mean()*100:.2f}%")
print(f"Churn rate in test: {y_test.mean()*100:.2f}%")
print(f"\nFeatures: {len(feature_cols)}")
print(f"Features list: {feature_cols}")

## 4. Train Multiple Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
}

# Train and evaluate models
model_results = {}
trained_models = {}

print("Training models...\n")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    model_results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    trained_models[name] = model
    
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  ROC AUC:   {roc_auc:.4f}\n")

# Model comparison
results_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'Accuracy': [model_results[m]['accuracy'] for m in model_results.keys()],
    'Precision': [model_results[m]['precision'] for m in model_results.keys()],
    'Recall': [model_results[m]['recall'] for m in model_results.keys()],
    'F1 Score': [model_results[m]['f1'] for m in model_results.keys()],
    'ROC AUC': [model_results[m]['roc_auc'] for m in model_results.keys()]
}).sort_values('ROC AUC', ascending=False)

print("\n" + "="*70)
print("MODEL PERFORMANCE COMPARISON")
print("="*70)
print(results_df.to_string(index=False))

# Best model
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"\n✓ Best Model: {best_model_name} (ROC AUC: {results_df.iloc[0]['ROC AUC']:.4f})")

## 5. Model Evaluation - ROC Curves & Performance

In [ ]:
# ROC Curve comparison
fig = go.Figure()

for model_name in model_results.keys():
    fpr, tpr, _ = roc_curve(y_test, model_results[model_name]['y_pred_proba'])
    roc_auc = auc(fpr, tpr)
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f"{model_name} (AUC = {roc_auc:.3f})",
        line=dict(width=2)
    ))

# Add diagonal
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random (AUC = 0.5)',
    line=dict(dash='dash', color='gray', width=2)
))

fig.update_layout(
    title='ROC Curve Comparison',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=500,
    hovermode='closest'
)
fig.show()

# Confusion matrices for best model
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (model_name, model) in enumerate(trained_models.items()):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    if model_name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained', 'Churned'])
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(f'{model_name}\nAccuracy: {accuracy_score(y_test, y_pred):.3f}')

plt.tight_layout()
plt.show()

# Best model detailed classification report
print("\n" + "="*70)
print(f"DETAILED RESULTS - {best_model_name.upper()}")
print("="*70)

if best_model_name == 'Logistic Regression':
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Retained', 'Churned']))

## 6. Feature Importance - What Drives Churn Predictions?

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(f"\nTop 15 Features ({best_model_name}):")
    print(feature_importance.head(15).to_string(index=False))
    
    # Visualize
    fig = px.bar(
        feature_importance.head(15),
        x='Importance',
        y='Feature',
        orientation='h',
        title=f'Top 15 Features for Churn Prediction ({best_model_name})',
        labels={'Importance': 'Importance Score', 'Feature': 'Feature'},
        color='Importance',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
    fig.show()
else:
    # For Logistic Regression, use coefficients
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': np.abs(best_model.coef_[0])
    }).sort_values('Importance', ascending=False)
    
    print(f"\nTop 15 Features ({best_model_name} - Absolute Coefficients):")
    print(feature_importance.head(15).to_string(index=False))
    
    fig = px.bar(
        feature_importance.head(15),
        x='Importance',
        y='Feature',
        orientation='h',
        title=f'Top 15 Features for Churn Prediction ({best_model_name})',
        labels={'Importance': 'Absolute Coefficient', 'Feature': 'Feature'},
        color='Importance',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
    fig.show()

## 7. Create Risk Scores & Customer Segmentation

In [ ]:
# Score all customers with best model
if best_model_name == 'Logistic Regression':
    df['churn_probability'] = best_model.predict_proba(scaler.transform(X))[:, 1]
else:
    df['churn_probability'] = best_model.predict_proba(X)[:, 1]

# Create risk tiers based on churn probability
def get_risk_tier(prob):
    if prob >= 0.5:
        return 'Critical'
    elif prob >= 0.35:
        return 'High'
    elif prob >= 0.20:
        return 'Medium'
    else:
        return 'Low'

df['risk_tier'] = df['churn_probability'].apply(get_risk_tier)

# Risk segmentation stats
risk_summary = df.groupby('risk_tier').agg({
    'customerID': 'count',
    'churn_probability': ['mean', 'min', 'max'],
    'Churn': ['sum', 'mean'],
    'MonthlyCharges': 'sum'
}).round(3)
risk_summary.columns = ['Count', 'Avg_Prob', 'Min_Prob', 'Max_Prob', 'Churned', 'Actual_Churn_Rate', 'Monthly_Revenue']

# Sort by risk
risk_order = ['Critical', 'High', 'Medium', 'Low']
risk_summary = risk_summary.reindex(risk_order)

print("\n" + "="*100)
print("CUSTOMER RISK SEGMENTATION")
print("="*100)
print(risk_summary)

# Visualize risk distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribution of Churn Probability", "Risk Tier Breakdown"),
    specs=[[{"type": "histogram"}, {"type": "pie"}]]
)

fig.add_trace(
    go.Histogram(x=df['churn_probability'], nbinsx=30, name='Churn Probability', 
                 marker_color='#3498db'),
    row=1, col=1
)

risk_counts = df['risk_tier'].value_counts().reindex(risk_order)
colors = {'Critical': '#e74c3c', 'High': '#f39c12', 'Medium': '#f1c40f', 'Low': '#2ecc71'}
fig.add_trace(
    go.Pie(labels=risk_counts.index, values=risk_counts.values, 
           marker=dict(colors=[colors[tier] for tier in risk_counts.index]),
           name='Risk Tier'),
    row=1, col=2
)

fig.update_layout(height=400, title_text="Churn Risk Distribution")
fig.show()

# High-risk customers
high_risk = df[df['risk_tier'].isin(['Critical', 'High'])].copy()
print(f"\n\nHIGH-RISK CUSTOMERS ({len(high_risk):,} customers)")
print(f"Avg Churn Probability: {high_risk['churn_probability'].mean():.2%}")
print(f"Estimated Revenue at Risk: ${high_risk.loc[high_risk['Churn']==1, 'MonthlyCharges'].sum():,.0f}")
print(f"Actual Churn Rate: {high_risk['Churn'].mean():.1%}")

## 8. Targeted Intervention Strategy

In [ ]:
# Define interventions by risk profile
interventions = []

for idx, row in df.iterrows():
    risk_tier = row['risk_tier']
    tenure = row['tenure']
    contract = row['Contract']
    has_support = row['TechSupport'] == 'Yes' or row['OnlineSecurity'] == 'Yes'
    payment = row['PaymentMethod']
    monthly = row['MonthlyCharges']
    
    actions = []
    priority = None
    
    if risk_tier == 'Critical':
        priority = 1
        actions.append("URGENT: Proactive outreach by retention specialist")
        if tenure < 6:
            actions.append("- Launch first-6-months onboarding program")
        if contract == 'Month-to-month':
            actions.append("- Offer 30% discount on 1-year contract")
        if not has_support:
            actions.append("- Bundle Tech Support + Online Security (50% off)")
        if 'Electronic check' in payment:
            actions.append("- Auto-pay setup incentive ($10 credit)")
    
    elif risk_tier == 'High':
        priority = 2
        actions.append("HIGH PRIORITY: Personalized retention offer")
        if tenure < 12:
            actions.append("- Early customer success check-in")
        if contract == 'Month-to-month':
            actions.append("- Targeted 1-year contract discount (20-25%)")
        if not has_support:
            actions.append("- Service bundle promotion")
        if monthly > 80:
            actions.append("- Loyalty reward for high-value customer")
    
    elif risk_tier == 'Medium':
        priority = 3
        actions.append("MEDIUM: Email campaign + targeted offer")
        if not has_support:
            actions.append("- Service adoption campaign")
        actions.append("- General satisfaction survey")
    
    else:  # Low
        priority = 4
        actions.append("LOW: Standard retention communication")
        actions.append("- Quarterly engagement newsletter")
    
    interventions.append({
        'customerID': row['customerID'],
        'risk_tier': risk_tier,
        'churn_probability': row['churn_probability'],
        'tenure': tenure,
        'contract': contract,
        'monthly_charges': monthly,
        'priority': priority,
        'recommended_actions': ' | '.join(actions)
    })

interventions_df = pd.DataFrame(interventions)

# Show sample high-risk customers for targeting
print("\n" + "="*120)
print("TOP 10 HIGHEST-RISK CUSTOMERS - IMMEDIATE ACTION REQUIRED")
print("="*120)

top_risk = interventions_df.nlargest(10, 'churn_probability')[['customerID', 'churn_probability', 'tenure', 'contract', 'monthly_charges']]
for idx, row in top_risk.iterrows():
    print(f"\nCustomer: {row['customerID']} | Churn Prob: {row['churn_probability']:.1%} | Tenure: {row['tenure']}mo | Contract: {row['contract']} | Spend: ${row['monthly_charges']:.2f}/mo")
    print(f"  → Action: {interventions_df[interventions_df['customerID']==row['customerID']]['recommended_actions'].values[0]}")

# Impact analysis
print("\n" + "="*120)
print("POTENTIAL IMPACT OF INTERVENTIONS")
print("="*120)

for tier in risk_order:
    tier_data = df[df['risk_tier'] == tier]
    revenue_at_risk = tier_data['MonthlyCharges'].sum() * tier_data['churn_probability'].mean()
    
    if tier == 'Critical':
        retention_lift = 0.40  # Expect 40% reduction in churn
        cost_per_customer = 50  # Cost to execute intervention
    elif tier == 'High':
        retention_lift = 0.25
        cost_per_customer = 20
    elif tier == 'Medium':
        retention_lift = 0.15
        cost_per_customer = 5
    else:
        retention_lift = 0.05
        cost_per_customer = 2
    
    total_cost = len(tier_data) * cost_per_customer
    revenue_saved = revenue_at_risk * retention_lift
    roi = (revenue_saved - total_cost) / total_cost if total_cost > 0 else 0
    
    print(f"\n{tier} Risk Tier ({len(tier_data):,} customers)")
    print(f"  Monthly Revenue at Risk: ${revenue_at_risk:,.0f}")
    print(f"  Intervention Cost: ${total_cost:,.0f}")
    print(f"  Expected Revenue Saved: ${revenue_saved:,.0f}")
    print(f"  ROI: {roi:.1%}")

## 9. Export Customer Risk Scores

In [ ]:
# Create comprehensive customer scoring dataset
customer_scores = df[[
    'customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 
    'Contract', 'PaymentMethod', 'InternetService',
    'TechSupport', 'OnlineSecurity', 'Churn'
]].copy()

customer_scores['churn_probability'] = df['churn_probability'].values
customer_scores['risk_tier'] = df['risk_tier'].values
customer_scores['engagement_score'] = df_model['engagement_score'].values

# Merge with interventions
customer_scores = customer_scores.merge(
    interventions_df[['customerID', 'recommended_actions']],
    on='customerID'
)

# Sort by risk
customer_scores['churn_probability'] = customer_scores['churn_probability'].round(4)
customer_scores = customer_scores.sort_values('churn_probability', ascending=False)

# Save to CSV
output_path = '../reports/customer_churn_risk_scores.csv'
customer_scores.to_csv(output_path, index=False)
print(f"✓ Exported {len(customer_scores):,} customer risk scores to: {output_path}")

# Summary for operations
print(f"\n{'CUSTOMER RISK SCORES - QUICK SUMMARY':^100}")
print("="*100)
print(f"\nTotal Customers: {len(customer_scores):,}")
print(f"Critical Risk: {len(customer_scores[customer_scores['risk_tier']=='Critical']):,} ({len(customer_scores[customer_scores['risk_tier']=='Critical'])/len(customer_scores)*100:.1f}%)")
print(f"High Risk: {len(customer_scores[customer_scores['risk_tier']=='High']):,} ({len(customer_scores[customer_scores['risk_tier']=='High'])/len(customer_scores)*100:.1f}%)")
print(f"Target for Intervention: {len(customer_scores[customer_scores['risk_tier'].isin(['Critical', 'High'])]):,} customers")

# Show sample output
print("\nSample Output (Top 5 Risk Customers):")
print(customer_scores[['customerID', 'churn_probability', 'risk_tier', 'tenure', 
                       'Contract', 'MonthlyCharges']].head(5).to_string(index=False))

# Save the best model
model_path = '../reports/best_churn_model.pkl'
joblib.dump(best_model, model_path)
print(f"\n✓ Saved best model ({best_model_name}) to: {model_path}")

# Save scaler for future predictions
scaler_path = '../reports/feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✓ Saved scaler to: {scaler_path}")

## 10. Summary & Key Takeaways

### Model Performance
- **Best Model:** Selected based on ROC AUC score (measures overall discrimination ability)
- **Key Metrics:** Precision (avoid false positives), Recall (catch real churners), F1 Score (balance), ROC AUC (overall performance)

### Customer Risk Segmentation
**Critical Risk (Highest Priority):**
- Immediate outreach by retention specialists
- Personalized multi-channel engagement
- Aggressive retention offers (30% discount on upgrades)
- Expected 40% reduction in churn with intervention

**High Risk (Priority Action):**
- Proactive support team outreach
- Targeted service bundle promotions
- Loyalty rewards for high-value customers
- Expected 25% reduction in churn with intervention

**Medium Risk (Passive Engagement):**
- Email campaigns and surveys
- Service adoption nudges
- General satisfaction touchpoints
- Expected 15% reduction in churn with intervention

**Low Risk (Nurture):**
- Standard retention communications
- Engagement newsletters
- Cross-sell/upsell opportunities
- Expected 5% reduction in churn with intervention

### Intervention ROI
Each risk tier has different intervention strategies and expected ROI. See section 8 for detailed financial impact.

### Next Steps
1. **Export Risk Scores** → Share with retention/support teams (customer_churn_risk_scores.csv)
2. **Launch Campaigns** → Execute interventions by risk tier in priority order
3. **Track Results** → Monitor actual churn vs predictions to validate model
4. **Iterate** → Retrain model quarterly with updated data and results
5. **Scale** → Automate scoring for new customers at signup

---

### Files Generated
- `customer_churn_risk_scores.csv` - All customers scored with risk tier and recommendations
- `best_churn_model.pkl` - Trained model for scoring new customers
- `feature_scaler.pkl` - Feature scaler for consistent preprocessing

**Ready to deploy retention campaigns! 🚀**